# PathSnap AM322 on Kaggle or Google Colab

**Kaggle:** Import this notebook into a private CPU notebook, enable Internet for the GitHub clone and dependency installation, and attach `Student folder.zip` as a private dataset with **Add Input**. The notebook accepts the archive or its extracted `train` and `student_test` folders. Run all cells, then download `am322-results.zip` from Output.

**Colab:** Open this notebook in a CPU runtime, and run all cells. Colab loads the ZIP from GitHub if you have not uploaded a copy.

The run uses the team-selected LightGBM classifier, 50 Optuna TPE trials with trip-grouped cross-validation, and a small decoder parameter sweep on training predictions. The 10 student-test trips are used for final evaluation.

In [ ]:
%pip -q install optuna lightgbm pyarrow pyproj shapely networkx scipy pyyaml

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import shutil
import subprocess
import sys

kaggle_input = Path("/kaggle/input")
on_kaggle = kaggle_input.is_dir()
runtime = Path("/tmp") if on_kaggle else Path("/content")
output_root = Path("/kaggle/working") if on_kaggle else runtime
repo = runtime / "ML_hackathon_Team-080"

if on_kaggle:
    raw_roots = sorted(p for p in kaggle_input.rglob("train") if p.is_dir() and (p.parent / "student_test").is_dir())
    archives = sorted(kaggle_input.rglob("Student folder.zip"))
    if len(raw_roots) == 1:
        dataset_root = raw_roots[0].parent
        archive = None
    elif not raw_roots and len(archives) == 1:
        dataset_root = None
        archive = archives[0]
    else:
        raise ValueError("Attach one private PathSnap dataset containing Student folder.zip or the train and student_test folders.")
else:
    dataset_root = None
    archive = runtime / "Student folder.zip"

if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/Vocod3r/ML_hackathon_Team-080.git", str(repo)], check=True)
if not on_kaggle and not archive.is_file():
    for ref in ("origin/main", "origin/p3-cloud-notebook"):
        spec = f"{ref}:Student folder.zip"
        if subprocess.run(
            ["git", "-C", str(repo), "cat-file", "-e", spec], capture_output=True
        ).returncode == 0:
            temporary = archive.with_suffix(".part")
            with temporary.open("wb") as destination:
                subprocess.run(["git", "-C", str(repo), "show", spec], stdout=destination, check=True)
            temporary.replace(archive)
            break
    else:
        raise FileNotFoundError("Upload Student folder.zip to /content, then rerun this cell.")

subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "4bab52b"], check=True)

raw = dataset_root or repo / "data/raw"
if archive is not None:
    raw.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as dataset:
        for member in dataset.infolist():
            parts = Path(member.filename).parts
            if len(parts) != 4 or parts[0] != "Student folder" or parts[1] not in {"train", "student_test"}:
                continue
            partition, trip, filename = parts[1:]
            if not (len(trip) == 8 and trip.isdecimal() and filename in {f"{trip}.{ext}" for ext in ("track", "nodes", "arcs", "route")}):
                continue
            target = raw / partition / trip / filename
            target.parent.mkdir(parents=True, exist_ok=True)
            with dataset.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)

for partition, count in [("train", 80), ("student_test", 10)]:
    assert len(list((raw / partition).glob("*/*.route"))) == count, partition
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

In [ ]:
from src.data.build_dataset import labelled_table
from src.data.io import load_trip
import pyarrow as pa
import pyarrow.parquet as pq

processed = repo / "data/processed"
for partition in ("train", "student_test"):
    target = processed / f"{partition}_candidates.parquet"
    if target.exists():
        continue
    temporary = target.with_suffix(".part")
    writer = None
    try:
        for trip_dir in sorted((raw / partition).iterdir()):
            table = pa.Table.from_pandas(labelled_table(load_trip(trip_dir)), preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema)
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(target)
    print(partition, pq.ParquetFile(target).metadata.num_rows)

In [ ]:
import gc
import pandas as pd
import yaml
from threadpoolctl import threadpool_limits
from src.tuning.bayes_opt import _with_team_model, tune

output = runtime / "am322-output"
hpo = output / "hpo"
artifacts = output / "artifacts"
cfg = _with_team_model(
    yaml.safe_load(Path("configs/hpo.yaml").read_text()),
    Path("results/models/selected_model.csv"),
    Path("configs/models.yaml"),
)
assert cfg["team_model"] == "lightgbm" and cfg["trials"] == 50
features = pd.read_csv(processed / "feature_cols.csv", header=None)[0].tolist()
if not (hpo / "comparison.csv").exists():
    train = pd.read_parquet(processed / "train_candidates.parquet")
    test = pd.read_parquet(processed / "student_test_candidates.parquet")
    with threadpool_limits(limits=min(4, os.cpu_count() or 1)):
        display(tune(train, features, cfg, hpo, test, artifacts))
    del train, test
    gc.collect()
else:
    display(pd.read_csv(hpo / "comparison.csv"))

In [ ]:
import json
import numpy as np
from src.hmm.evaluate_hmm import evaluate

hmm = output / "hmm"
hmm.mkdir(parents=True, exist_ok=True)
decoder_cfg = yaml.safe_load(Path("configs/hmm.yaml").read_text())
columns = ["trip_id", "obs_idx", "t", "lon", "lat", "arc_id", "arc_len", "frac", "ref_arc", "probability"]
trip_ids = sorted(pd.read_parquet(hpo / "oof_probabilities.parquet", columns=["trip_id"])["trip_id"].unique())
# Fixed eight-trip subset keeps decoder tuning tractable.
subset = sorted(np.random.default_rng(322).choice(trip_ids, size=8, replace=False))
validation = pd.read_parquet(hpo / "oof_probabilities.parquet", columns=columns, filters=[("trip_id", "in", subset)])
records = []
for beta in (15.0, 30.0, 60.0):
    trial_cfg = decoder_cfg | {"beta_m": beta}
    metrics = evaluate(validation, raw / "train", trial_cfg, "probability")
    decoded = metrics.loc[metrics["method"].eq("hmm")]
    records.append({"beta_m": beta, "pointwise_accuracy": decoded["pointwise_accuracy"].mean(),
                    "continuity": decoded["continuity"].mean()})
ranking = pd.DataFrame(records).sort_values(["pointwise_accuracy", "continuity"], ascending=False)
ranking.to_csv(hmm / "decoder_sweep.csv", index=False)
(hmm / "decoder_training_trips.json").write_text(json.dumps(subset, indent=2))
decoder_cfg["beta_m"] = float(ranking.iloc[0]["beta_m"])
(artifacts / "hmm_config.json").write_text(json.dumps(decoder_cfg, indent=2))

test_scores = pd.read_parquet(hpo / "student_test_probabilities.parquet", columns=columns)
results = evaluate(test_scores, raw / "student_test", decoder_cfg, "probability")
results.to_csv(hmm / "student_test_comparison.csv", index=False)
display(results.groupby("method")[["pointwise_accuracy", "continuity", "sequence_exact_match"]].mean())

In [ ]:
bundle = output_root / "am322-results.zip"
with ZipFile(bundle, "w", compression=8) as result_zip:
    for path in sorted(output.rglob("*")):
        if path.is_file() and path.suffix != ".parquet":
            result_zip.write(path, path.relative_to(output))
if on_kaggle:
    print(f"Download {bundle.name} from the Kaggle Output panel.")
else:
    from google.colab import files

    files.download(str(bundle))